#Chapter 5: Query Plans - Fundamentals


##Sample data - Set up


In [0]:
%sql
-- Create catalog if it doesn't exist
CREATE CATALOG IF NOT EXISTS sparkuiessentials;

-- Create schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS sparkuiessentials.demos;

In [0]:
%sql

DROP TABLE IF EXISTS sparkuiessentials.demos.mock_transactions;

CREATE TABLE sparkuiessentials.demos.mock_transactions(
    TranId STRING NOT NULL, 
    TranAmount DECIMAL(18,2), 
    TranDate DATE, 
    Region STRING
);

-- Data file 1 (TranAmounts: 5.00 - 75.20)
INSERT INTO sparkuiessentials.demos.mock_transactions
VALUES
    ("T01", 20.10, "2026-08-01", "East"),
    ("T02", 40.00, "2026-08-02", "East"),
    ("T03", 45.50, "2026-08-03", "East"),
    ("T04", 25.00, "2026-08-04", "East"),
    ("T05", 64.70, "2026-08-05", "East"),
    ("T06", 12.50, "2026-08-06", "East"),
    ("T07", 5.00, "2026-08-07", "East"),
    ("T08", 75.20, "2026-08-08", "East"),
    ("T09", 20.00, "2026-08-09", "East"),
    ("T10", 50.15, "2026-08-10", "East"),
    ("T11", 23.05, "2026-08-11", "East"),
    ("T12", 17.75, "2026-08-12", "East"),
    ("T13", 42.90, "2026-08-13", "East"),
    ("T14", 28.65, "2026-08-14", "East")
    AS t(TranId, TranAmount, TranDate, Region);

-- Data file 2 (TranAmounts: 54:00 - 95.45)
INSERT INTO sparkuiessentials.demos.mock_transactions
VALUES
    ("T15", 55.55, "2026-08-01", "West"),
    ("T16", 68.80, "2026-08-02", "West"),
    ("T17", 54.00, "2026-08-03", "West"),
    ("T18", 95.45, "2026-08-04", "West"),
    ("T19", 80.20, "2026-08-05", "West"),
    ("T20", 77.20, "2026-08-06", "West"),
    ("T21", 65.90, "2026-08-07", "West"),
    ("T22", 85.75, "2026-08-08", "West")
    AS t(TranId, TranAmount, TranDate, Region);    

-- Data file 3 (TranAmounts: 125.00 - 144.05)
INSERT INTO sparkuiessentials.demos.mock_transactions
VALUES
    ("T23", 125.00, "2026-08-03", "North"),
    ("T24", 144.05, "2026-08-04", "North")
    AS t(TranId, TranAmount, TranDate, Region);

-- Data file 4 (TranAmounts: 154.00 - 190.00)
INSERT INTO sparkuiessentials.demos.mock_transactions
VALUES
    ("T25", 162.15, "2026-08-09", "South"),
    ("T26", 154.00, "2026-08-10", "South"),
    ("T27", 190.00, "2026-08-11", "South"),
    ("T28", 170.50, "2026-08-12", "South")
    AS t(TranId, TranAmount, TranDate, Region);


In [0]:
%sql

ANALYZE TABLE sparkuiessentials.demos.mock_transactions COMPUTE STATISTICS FOR ALL COLUMNS;

In [0]:
%sql

-- numFiles: 4
DESCRIBE DETAIL sparkuiessentials.demos.mock_transactions;

##1.0 Logical & Physical Query Plans - as in the Illustrations

In [0]:
%sql
SELECT 
    c.c_custkey,
    c.c_acctbal,
    o.o_orderkey,
    o.o_orderdate
FROM samples.tpch.orders o
INNER JOIN samples.tpch.customer c
    ON o.o_custkey = c.c_custkey
WHERE c.c_mktsegment IN ('BUILDING', 'MACHINERY', 'AUTOMOBILE')

###Query Plans - EXTENDED mode

<img src="./Illustrations/ch05_logical_plans v0.1.png" alt="Illustrations/ch05_logical_plans.png"/>


In [0]:
%sql
EXPLAIN EXTENDED
SELECT 
    c.c_custkey,
    c.c_acctbal,
    o.o_orderkey,
    o.o_orderdate
FROM samples.tpch.orders o
INNER JOIN samples.tpch.customer c
    ON o.o_custkey = c.c_custkey
WHERE c.c_mktsegment IN 
    (   
        'BUILDING', 
        'MACHINERY', 
        'AUTOMOBILE'
    )

###Tips: extended mode vs formatted mode
*Tip*: The physical plan generated in **extended** mode is not so easy to follow. 

So, you might want to look at the **formatted** physical plan. (e.g. `df.explain(mode=“formatted”)`). 


###Query Plan - FORMATTED mode (Physical plan only)

**FORMATTED** mdoe generates two sections: a physical plan outline and node details

<img src="./Illustrations/ch05_physical_plans v0.1.png" alt="Illustrations/ch05_physical_plans.png"/>


In [0]:
%sql
EXPLAIN FORMATTED
SELECT 
    c.c_custkey,
    c.c_acctbal,
    o.o_orderkey,
    o.o_orderdate
FROM samples.tpch.orders o
INNER JOIN samples.tpch.customer c
    ON o.o_custkey = c.c_custkey
WHERE c.c_mktsegment IN 
    (   
        'BUILDING', 
        'MACHINERY', 
        'AUTOMOBILE'
    )

##Quick tour of Physical plan / SQL DAG in Spark UI


##Java code generated by WholeStageCodegen

A quick look at Java code generated by WholeStageCodegen:

Troubleshooting info:

When you run the SQL/Dataframe to generate Java code using WholestageCodegen, if the output says **Found 0 WholeStageCodegen subtrees** — so the cluster is processing the command, but no codegen subtrees are being generated.

To demonstrate EXPLAIN CODEGEN, switch to a Single User (Assigned) access mode classic cluster. That gives you direct JVM access where WholeStageCodegen runs natively, and the generated Java code will appear in the output.

Another likely cause is that the samples catalog (`samples.tpch.*`) tables are served through a shared/federated read path that bypasses WholeStageCodegen, so EXPLAIN CODEGEN finds no subtrees. . 


In [0]:
%sql
EXPLAIN CODEGEN
SELECT TranId, TranAmount
FROM sparkuiessentials.demos.mock_transactions
WHERE TranAmount >= 80.00

##Tips: How to Download SQL DAGs and Phyiscal plans from Spark UI

* **Download from the SQL tab --> Query Details page** — captures a snapshot of what is currently displayed.
* **What you see is what you get** — expand/collapse sections and DAG nodes before downloading.
* Choose what to capture:
    - **SQL DAG** — collapse nodes and other sections you don’t need.
    - **SQL DAG + runtime metrics** — expand only the nodes of interest.
    - **Physical Plan** — expand the Physical Plan and collapse other sections.
* Keep the cluster running when downloading if you need runtime metrics.

**Key takeaway**: Prepare the Query Details view first, then download the snapshot you want to analyse or retain.


##2.0 Pushdowns: File Pruning and Column Pruning

Demonstrates: 
* *Predicate Pushdown --> File pruning*
    - Note: Predicate Pushdown does more things than just File pruning:
    - File-level pruning — Skip entire files based on min/max statistics in file footers
    - Row group pruning — Skip row groups within files (Parquet-specific)
    - Row-level filtering — Filter individual rows during the scan
* *Projection Pushdown --> Column pruning*

In [0]:
from pyspark.sql.functions import col, lit, initcap
df_narrowTransf = (

    spark.read.table("sparkuiessentials.demos.mock_transactions")
    .select("TranId", "TranAmount")
    .where((col("TranAmount") >= lit("80.00").cast("decimal(18,2)"))
            & (col("Region").isin("East", "West")))
)

display(df_narrowTransf)

In [0]:
df_narrowTransf = (
    spark.read.table("sparkuiessentials.demos.mock_transactions")
    .select("*")
)

display(df_narrowTransf)

##3. PITFALLS: Predicate Pushdown


###Example 1 - explicit / implicit casting of literals

In [0]:
from pyspark.sql.functions import col, lit, initcap, concat
df_result = (

    spark.read.table("sparkuiessentials.demos.mock_transactions")
    .select("TranId", "TranAmount")
    .where((col("TranAmount") >= lit("80.00").cast("decimal(18,2)")))
)

display(df_result)

In [0]:
from pyspark.sql.functions import col, lit
df_result = (

    spark.read.table("sparkuiessentials.demos.mock_transactions")
    .select("TranId", "TranAmount")
    .where(col("TranAmount") >= 80.00)
)

display(df_result)

In [0]:
%sql
SELECT TranId, TranAmount
FROM sparkuiessentials.demos.mock_transactions
WHERE TranAmount >= CAST('80.00' AS DECIMAL(18,2))

In [0]:
%sql
SELECT TranId, TranAmount
FROM sparkuiessentials.demos.mock_transactions
WHERE TranAmount >= 80.00

In [0]:
df_result = (
    spark.read.table("sparkuiessentials.demos.mock_transactions")
    .select("TranId", "TranAmount", "Region")
    .where(col("TranAmount").cast("double") >= 80.00) 
)

display(df_result)

###Example 2: Functions on columns in Filter condition

In [0]:
df_result = (
    spark.read.table("sparkuiessentials.demos.mock_transactions")
    .select("TranId", "TranAmount", "Region")
    .where(col("Region").isin("North", "South")) 
)

display(df_result)

In [0]:
from pyspark.sql.functions import upper, col
df_result = (
    spark.read.table("sparkuiessentials.demos.mock_transactions")
    .select("TranId", "TranAmount", "Region")
    .where(upper(col("Region")).isin("NORTH", "SOUTH"))
)

display(df_result)

In [0]:
df_result = (
    spark.read.table("sparkuiessentials.demos.mock_transactions")
    .select("TranId", "TranAmount", "Region")
    .where(col("Region").isin(initcap(lit("north")), initcap(lit("south")))) 
)

display(df_result)

In [0]:
from pyspark.sql.functions import upper, col, concat_ws
df_result = (
    spark.read.table("sparkuiessentials.demos.mock_transactions")
    .select("TranId", "TranAmount", "Region")
    .where(concat_ws(": ", lit("Region"), col("Region")).isin(lit("Region: North"), lit("Region: South")))
)

display(df_result)

###Example 3: Expressions on columns in Filter condition

In [0]:
df_result = (
    spark.read.table("sparkuiessentials.demos.mock_transactions")
    .select("TranId", "TranAmount", "Region")
    .where(col("TranAmount") * 1.10 >= 88.00) 
)

display(df_result)